### **House Price Prediction with MLflow:**

In this project, we will:

* Run a hyperparameter tuning while training the model.
* Log every hyperparameter and metrics in the MLflow UI.
* Compare the results of the various runs in the MLflow UI.
* Choose the best run and register it as a model. 

In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import fetch_california_housing


In [2]:
housing = fetch_california_housing()

print(housing)

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8)), 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)), 'frame': None, 'target_names': ['MedHouseVal'], 'feature_names': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'], 'DESCR': '.. _california_housing_dataset

In [3]:
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

In [4]:
#Preparing the dataset properly
data = pd.DataFrame(housing.data, columns=housing.feature_names)

data['Price'] = housing.target

data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### **Train Test Split, Model Hyperparameter Tuning, MLflow Experiments:**

In [5]:
from urllib.parse import urlparse
#urlparse is a function from Python’s urllib.parse module 
#that helps in breaking down a URL into its different components.

data.shape

(20640, 9)

In [6]:
#Independent and  Dependent Features
x = data.drop(columns='Price')

y = data['Price']

In [8]:
#Splitting data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [9]:
from mlflow.models import infer_signature

signature = infer_signature(x_train, y_train)

In [10]:
#Defining the hyperparameter grid
params_grid = {
    'n_estimators' : [100, 200], 
    'max_depth' : [5, 10, None],
    'min_samples_split' : [2, 5], 
    'min_samples_leaf' : [1, 2]
}

In [12]:
#Hyperparameter tuning using GridSearchCV
def hyperparameter_tuning(x_train, y_train, params_grid):
    rf = RandomForestRegressor()
    grid = GridSearchCV(estimator = rf, param_grid = params_grid, cv=3, n_jobs=-1, verbose=2, 
                        scoring = 'neg_mean_squared_error')
    grid.fit(x_train, y_train)
    return grid

In [13]:
#Startgin the MLflow Experiment
with mlflow.start_run():
    #Hyperparameter tuning
    grid = hyperparameter_tuning(x_train, y_train, params_grid)

    #Getting the best parameters
    best_model = grid.best_estimator_

    #Evaluating the best mdoel
    y_pred = best_model.predict(x_test)
    mse = mean_squared_error(y_test, y_pred)

    #Logging the best parameters and the evaluation metrics
    mlflow.log_param("best_n_estimators", grid.best_params_['n_estimators'])
    mlflow.log_param("best_max_depth", grid.best_params_['max_depth'])
    mlflow.log_param("best_min_samples_split", grid.best_params_['min_samples_split'])
    mlflow.log_param("best_min_samples_leaf", grid.best_params_['min_samples_leaf'])
    mlflow.log_metric("mse", mse)

    #Tracking url
    mlflow.set_tracking_uri(uri = 'http://127.0.0.1:5000')
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_url_type_store != "file":
    #When .scheme extracts as 'file', it means that 
    #MLflow is tracking and storing experiments locally on your file system, rather than using a remote server.
        mlflow.sklearn.log_model(best_model, "model", registered_model_name="Best RandomForest Model")
    else:
        mlflow.sklearn.log_model(best_model, "model", signature=signature)

    print(f"Best Hyperparameters: {grid.best_params_}")
    print(f"Mean Squared Error: {mse}")

Fitting 3 folds for each of 24 candidates, totalling 72 fits


2025/02/06 11:19:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'Best RandomForest Model'.
2025/02/06 11:19:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best RandomForest Model, version 1
Created version '1' of model 'Best RandomForest Model'.


Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Mean Squared Error: 0.25072417368226346
🏃 View run upbeat-bee-480 at: http://127.0.0.1:5000/#/experiments/0/runs/13474f5fe45248cf954c87ef0c58d69a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


### **Important Point:**

**mlflow.sklearn.log_model(best_model, "model", registered_model_name="Best RandomForest Model"):**

Why is signature not used here?
This version registers the model in MLflow’s Model Registry (registered_model_name="Best RandomForest Model").
When registering a model in a remote tracking server, the model is stored in an artifact repository, and MLflow automatically infers the signature when it is logged.
The Model Registry keeps track of model versions, allowing easy deployment and version control.

**mlflow.sklearn.log_model(best_model, "model", signature=signature):
**

Why is signature used here?
When logging locally (file storage), MLflow does not always infer the input-output schema automatically.
The signature explicitly defines the expected input and output format of the model (e.g., data types, column names, etc.).
This is useful if you want to ensure consistency when serving the model later.
